# Tutorial 11: Graph Neural Network Forward Model

In this tutorial we build a **Graph Neural Network (GNN)** that predicts Hamiltonian parameters
($f_q$, $\alpha$, $f_r$, $\kappa$, $g$) directly from the *design* geometry of a qubit–cavity
circuit — the **forward modelling** direction.

### Why a graph?

| Old approach (Tutorial 8) | New approach (this tutorial) |
|---|---|
| Flat-vector MLP, fixed input shape | Graph-based, variable input shape |
| One component type only | Heterogeneous components via sub-encoders |
| Components treated in isolation | Message-passing captures parasitic loading |
| No transfer learning | Architecture is transfer-learning ready |

We model each quantum circuit as a small **graph**:
- **Nodes** = components (qubit, claw, resonator, …) with heterogeneous design features
- **Edges** = physical connections (capacitive coupling, CPW junctions)

A `NodeEncoder` projects each component's variable-length features into a fixed-size latent
vector, then `GCNConv` layers perform message passing, and a readout head predicts the
Hamiltonian targets.

### Prerequisites

```bash
pip install SQuADDS[graph]   # installs tensorflow + spektral
```

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib inline

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from squadds.ml.graph.featurizer import (
    build_vocab,
    CircuitGraphBuilder,
    SQuADDSGraphDataset,
)
from squadds.ml.graph.gnn_model import GraphForwardModel
from squadds.ml.graph.trainer import GraphTrainer, plot_predictions

import tensorflow as tf
print(f"TensorFlow {tf.__version__}")

import spektral
print(f"Spektral {spektral.__version__}")

## 2. Load Training Data from SQuADDS

We re-use the same qubit–cavity training data from **Tutorial 8** (`training_data.parquet`).
Each row describes a `TransmonCross` + `CavityClaw` coupled system with:

| Column group | Columns |
|---|---|
| **Design parameters** | `cross_length`, `cross_gap`, `ground_spacing`, `claw_length`, `coupling_length`, `total_length` |
| **Hamiltonian targets** | `qubit_frequency_GHz`, `anharmonicity_MHz`, `cavity_frequency_GHz`, `kappa_kHz`, `g_MHz` |
| **Derived** | `EC`, `EJ` |

For the **forward model** we go: design parameters → Hamiltonian targets.

In [ ]:
df = pd.read_parquet("data/training_data.parquet")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dataset: {len(df):,} rows × {df.shape[1]} columns")
df.head(3)

In [ ]:
# Define forward-model columns
HAMILTONIAN_TARGETS = [
    "qubit_frequency_GHz",
    "anharmonicity_MHz",
    "cavity_frequency_GHz",
    "kappa_kHz",
    "g_MHz",
]

# Design parameters that go into TRANSMONCross (the qubit node)
QUBIT_DESIGN_PARAMS = ["cross_length", "cross_gap", "ground_spacing"]

# Design parameters that go into CavityClaw (the cavity node)
CAVITY_DESIGN_PARAMS = ["claw_length", "coupling_length", "total_length"]

## 3. Build the Parameter-Key Vocabulary

The `GeometricEncoder` inside our GNN uses an `Embedding` layer to encode parameter
*key names* (like `"cross_length"`, `"claw_length"`) as learnable vectors. We first build
a vocabulary mapping each unique parameter name to an integer ID.

The `build_vocab()` function scans all 44 enriched component JSONs that ship with
`qiskit-metal` and collects every `parameter_name`.

In [ ]:
vocab = build_vocab()
print(f"Vocabulary size: {len(vocab)} keys")
print(f"Sample entries: {dict(list(vocab.items())[:8])}")

## 4. Convert Each Row into a Circuit Graph

Each row in our DataFrame describes a **two-component circuit**:

```
Node 0: TransmonCross  ──edge──  Node 1: CavityClaw
```

The `CircuitGraphBuilder` calls `ComponentFeaturizer` for each node to produce
a raw feature vector (layer stack, design params, area, perimeter, ports), then
packs them into a `spektral.data.Graph` object with the adjacency matrix and
the Hamiltonian target vector.

In [ ]:
K_MAX = 20  # max design params per node (padded)
builder = CircuitGraphBuilder(vocab=vocab, k_max=K_MAX)

graphs = []
for _, row in df.iterrows():
    # Build design-option dicts for each component
    qubit_opts = {k: f"{row[k]}um" for k in QUBIT_DESIGN_PARAMS}
    cavity_opts = {k: f"{row[k]}um" for k in CAVITY_DESIGN_PARAMS}

    # Hamiltonian targets
    targets = [row[t] for t in HAMILTONIAN_TARGETS]

    g = builder.build(
        components=[
            ("TransmonCross", qubit_opts),
            ("CavityClaw", cavity_opts),  # CavityClaw JSON may not exist — that's OK
        ],
        edges=[(0, 1)],       # qubit ↔ cavity physical connection
        targets=targets,
    )
    graphs.append(g)

print(f"Built {len(graphs):,} graphs")
print(f"Node feature dim: {graphs[0].x.shape[1]}")
print(f"Target dim: {graphs[0].y.shape[0]}")

### 4a. Inspect a Sample Graph

In [ ]:
sample = graphs[0]
print("Nodes x shape :", sample.x.shape)
print("Adjacency     :", sample.a.toarray())
print("Targets (y)   :", sample.y)
print()
print("Target mapping:")
for name, val in zip(HAMILTONIAN_TARGETS, sample.y):
    print(f"  {name:>25s} = {val:.4f}")

### 4b. Visualize the Graph Topology

Our circuit is a simple 2-node graph, but the architecture generalises to
larger multi-qubit circuits.

In [ ]:
try:
    import networkx as nx

    G = nx.Graph()
    G.add_node(0, label="TransmonCross")
    G.add_node(1, label="CavityClaw")
    G.add_edge(0, 1)

    fig, ax = plt.subplots(1, 1, figsize=(4, 3))
    pos = nx.spring_layout(G, seed=42)
    labels = nx.get_node_attributes(G, "label")
    nx.draw(
        G, pos, ax=ax, with_labels=True, labels=labels,
        node_color=["#4FC3F7", "#AED581"], node_size=2000,
        font_size=9, font_weight="bold", edge_color="#888", width=2,
    )
    ax.set_title("Qubit–Cavity Circuit Graph")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install networkx for graph visualization: pip install networkx")

## 5. Create Train / Validation / Test Splits

In [ ]:
dataset = SQuADDSGraphDataset(graphs, val_split=0.1, test_split=0.1, seed=42)
print(f"Train : {len(dataset.train_graphs):,}")
print(f"Val   : {len(dataset.val_graphs):,}")
print(f"Test  : {len(dataset.test_graphs):,}")

## 6. Build the Graph Forward Model

The model architecture:

```
Raw node features ──► NodeEncoder ──► E_static (N, 128)
                                          │
                                     GCNConv × 2  ←── adjacency
                                          │
                                   E_context (N, 128)
                                          │
                                   GlobalAttentionPool
                                          │
                                   graph embedding (128,)
                                          │
                                    Readout MLP
                                          │
                                   ŷ = [f_q, α, f_r, κ, g]
```

The `NodeEncoder` fuses three sub-encoders:
- **LayerStackEncoder** — Conv1D interface detector on the (5, 3) layer-stack matrix
- **GeometricEncoder** — DeepSets over variable-length design parameters
- **PortEncoder** — Dense on port-type counts

In [ ]:
model_builder = GraphForwardModel(
    vocab_size=len(vocab),
    embed_dim=32,
    node_latent_dim=128,
    n_gcn_layers=2,
    n_targets=len(HAMILTONIAN_TARGETS),
    k_max=K_MAX,
    readout_dim=64,
    dropout_rate=0.1,
)

# Peek at the model architecture
model = model_builder.build()
model.summary()

## 7. Train the Model

We use `GraphTrainer` which wraps Keras training with:
- **Adam** optimizer with MSE loss
- **ReduceLROnPlateau** learning-rate scheduling
- **EarlyStopping** on validation loss
- **DisjointLoader** for variable-size graph batching

In [ ]:
trainer = GraphTrainer(
    model_builder=model_builder,
    learning_rate=1e-3,
    target_names=HAMILTONIAN_TARGETS,
)

history = trainer.train(
    train_graphs=dataset.train_graphs,
    val_graphs=dataset.val_graphs,
    epochs=100,
    batch_size=64,
    patience=15,
    verbose=1,
)

### 7a. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["loss"], label="train")
if "val_loss" in history:
    axes[0].plot(history["val_loss"], label="val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].set_yscale("log")

axes[1].plot(history["mae"], label="train")
if "val_mae" in history:
    axes[1].plot(history["val_mae"], label="val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE")
axes[1].set_title("Mean Absolute Error")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Evaluate on the Test Set

We compute per-target **R²**, **RMSE**, and **MAE** on the held-out test set.

In [ ]:
metrics = trainer.evaluate(dataset.test_graphs)

results_df = pd.DataFrame(metrics).T
results_df.index.name = "Target"
results_df

### 8a. Parity Plots (Predicted vs. True)

In [ ]:
y_pred = trainer.predict(dataset.test_graphs)
y_true = np.array([g.y for g in dataset.test_graphs])

fig = plot_predictions(y_true, y_pred, target_names=HAMILTONIAN_TARGETS)
plt.show()

## 9. Transfer Learning Demo

A key advantage of the graph architecture is **transfer learning**. The
`GeometricEncoder` (DeepSets) handles *any* number of design parameters via
its vocabulary-based embedding — no fixed input shape.

Below we show that the *same trained model* can process a graph where one node
is a hypothetical `TransmonPocket` (different design keys, different count)
**without any code changes**.

In [ ]:
# A hypothetical TransmonPocket component with completely different design keys
pocket_opts = {
    "pad_width": "425um",
    "pad_height": "90um",
    "pad_gap": "30um",
    "pocket_width": "650um",
    "pocket_height": "650um",
}

# Build a graph with TransmonPocket (node 0) + CavityClaw (node 1)
transfer_graph = builder.build(
    components=[
        ("TransmonPocket", pocket_opts),  # different component!
        ("CavityClaw", {"claw_length": "200um", "coupling_length": "300um", "total_length": "3000um"}),
    ],
    edges=[(0, 1)],
)

print(f"TransmonPocket node features shape: {transfer_graph.x.shape}")
print("→ Same feature dimension as TransmonCross — the model can process it!")

# The model runs without error — it just hasn't been fine-tuned on this component yet
# In a real workflow, you would fine-tune on a small dataset of TransmonPocket circuits.

## 10. Save and Load the Model

In [ ]:
# Save
trainer.save("saved_models/graph_forward_model")
print("Model saved to saved_models/graph_forward_model/")

# Load
loaded_trainer = GraphTrainer.load("saved_models/graph_forward_model")
loaded_preds = loaded_trainer.predict(dataset.test_graphs[:5])
print(f"\nLoaded model predictions shape: {loaded_preds.shape}")
print(loaded_preds)

## Summary

| Step | What we did |
|---|---|
| **Data** | Loaded the same `training_data.parquet` from Tutorial 8 |
| **Featurization** | Converted each row into a 2-node circuit graph (TransmonCross ↔ CavityClaw) |
| **Model** | Built a `GraphForwardModel` with sub-encoders + GCN + attention pooling |
| **Training** | Trained end-to-end with MSE loss, early stopping, LR scheduling |
| **Evaluation** | Computed per-target R², RMSE, MAE on the test set with parity plots |
| **Transfer** | Demonstrated that the model handles different component types without re-architecture |
| **Serialization** | Saved and reloaded the model from disk |

### What's Next?

- **Scale up:** Extend to multi-qubit, multi-resonator circuits (3+ nodes)
- **Fine-tune:** Transfer-learn to new component geometries with small datasets
- **Edge features:** Encode coupling strengths, CPW impedances on edges
- **Inverse model:** Combine with an Analyzer-based search to go Hamiltonian → Design